## Forward modeling

The propagation of seismic waves in heterogeneous, isotropic, elastic earth media can be expressed by the elastodynamic equations:

\begin{equation}
\left\{\begin{array}{ll} 
\dfrac{\partial \sigma_{xx}}{\partial t} - (\lambda+2\mu)\dfrac{\partial v_{x}}{\partial x}-\lambda\dfrac{\partial v_{z}}{\partial z}=f_{\sigma_{xx}},\\
\dfrac{\partial\sigma_{zz}}{\partial t}-(\lambda+2\mu)\dfrac{\partial v_{z}}{\partial z} -\lambda\dfrac{\partial v_{x}}{\partial x}=f_{\sigma_{zz}},\\
\dfrac{\partial \sigma_{zx}}{\partial t} -\mu\Big(\dfrac{\partial v_{x}}{\partial z} + \dfrac{\partial v_{z}}{\partial x}\Big) = f_{\sigma_{zx}},\\
\rho\dfrac{\partial v_x}{\partial t} - \Big(\dfrac{\partial \sigma_{xx}}{\partial x} + \dfrac{\partial \sigma_{xz}}{\partial z} )=0,\\
\rho\dfrac{\partial v_z}{\partial t} - \Big(\dfrac{\partial \sigma_{zx}}{\partial x}+\dfrac{\partial \sigma_{zz}}{\partial z}\Big)=0 .
\end{array}\right.
\end{equation}


where $\vec{v}=(v_x,~v_z)$ are the horizontal and vertical particle velocity fields, $\sigma=(\sigma_{xx},~\sigma_{zz},~\sigma_{xz}$) are the stress fields, $f=$($f_{\sigma_{xx}},~f_{\sigma_{xx}})$
are the source terms, $\rho$ is density, $\lambda$ and $\mu$ are the Lame parameters. The elastic wave equation given described above can be organized using a compact formulation as presented by Chen and Sacchi (2020):

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-{\bf D} \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-{\bf C D}^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}

being ${\bf C}$ the isotropic elastic tensor in Voigt notation, ${\bf D}$ is a collection of spatial differential operators, defined as

\begin{equation}
\begin{split}
   {\bf C}=\left(\begin{array}{ccc}
   \lambda+2 \mu & \lambda & 0 \\
   \lambda & \lambda+2 \mu & 0 \\ 
   0 & 0 & \mu \end{array}\right)~~\text{e}~~
{\bf D}=\left(\begin{array}{ccc}  
\dfrac{\partial}{\partial x}& 0 &\dfrac{\partial}{\partial z} \\
0 &  \dfrac{\partial}{\partial z}&\dfrac{\partial}{\partial x}
\end{array}\right)  .
\end{split}
\end{equation}

In our compact formulation, the stress tensor $(\sigma)$ is represented is written in vectorial form:

\begin{equation}
\sigma=\left(\begin{array}{c}
 \sigma_{xx}  \\ 
 \sigma_{zz}  \\ 
 \sigma_{zx}
\end{array}\right)
\end{equation}

The conversion from matrix to vector form of the stress tensor is done using the vec() function. An example of this application is $\sigma$=vec($\sigma$).

In [ ]:
from examples.seismic.source import RickerSource, TimeAxis
from examples.seismic import setup_geometry, PointSource, Receiver
from examples.seismic import SeismicModel
from examples.seismic.stiffness.utils import C_Matrix, D, S, vec
from devito import (Eq, Operator, VectorTimeFunction, TensorTimeFunction,
                    VectorFunction, solve)
                    
import matplotlib.pyplot as plt
import numpy as np
import segyio
import cv2

from devito import configuration, norm
configuration['log-level'] = 'WARNING'

from examples.seismic.tutorials.lista_utils import *

from ipywidgets import interact, widgets

from utils import *

%matplotlib widget

In [ ]:
dir = './models'
model_pcs = load_marmousi(dir='./models', PCS=True)
model_pcs_han = load_marmousi(dir='./models', PCS='Han')
model_elastic = load_marmousi(dir='./models', PCS=False)

In [ ]:
plot_model(model_pcs, params=['Phi', 'cc', 'Sw'], figsize=(18,3))
plot_model(model_pcs_han, params=['Phi', 'cc', 'Sw'], figsize=(18,3))
plot_model(model_elastic, params=['vp', 'vs', 'rho'], figsize=(18,3))

In [ ]:
def get_Han(model):
    Phi = model.Phi.data
    cc = model.cc.data
    Sw = model.Sw.data

    rho_c, rho_q, rho_w, rho_h = 2.55, 2.65, 1, 0.1
    rho_m = cc * (rho_c - rho_q) + rho_q
    rho_f = Sw * (rho_w - rho_h) + rho_h
    rho = Phi * (rho_f - rho_m) + rho_m

    a1, a2, a3, b1, b2, b3 = 5.5, 6.9, 2.2, 3.4, 4.7, 1.8
    vp = a1 - a2 * Phi - a3 * cc
    vs = b1 - b2 * Phi - b3 * cc

    return [vp, vs, rho]

def get_VRH(model):
    Phi = model_pcs.Phi.data
    cc = model_pcs.cc.data
    Sw = model_pcs.Sw.data

    rho_c, rho_q, rho_w, rho_h = 2.55, 2.65, 1, 0.1
    K_c, K_q, K_w, K_h = 21, 37, 2.25, 0.04
    mu_c, mu_q = 10, 44

    rho_m = cc * (rho_c - rho_q) + rho_q
    rho_f = Sw * (rho_w - rho_h) + rho_h

    K_v = (1 - Phi) * (cc * (K_c - K_q) + K_q) + Phi * (Sw * (K_w - K_h) + K_h)
    K_r = 1 / ((1 -Phi) * cc / K_c + (1 - Phi) * (1 - cc) / K_q + Phi * Sw / K_w + Phi * (1 - Sw) / K_h)
    K_sat = (K_v + K_r) / 2

    mu_v = (1 - Phi) * (cc * (mu_c - mu_q) + mu_q)
    mu_sat = mu_v / 2

    rho = Phi * (rho_f - rho_m) + rho_m
    vp = ((K_sat + 4 / 3 * mu_sat) / rho)**(1/2)
    vs = (mu_sat / rho)**(1/2)

    return [vp, vs, rho]

def get_KT(model):
    Phi = model.Phi.data
    cc = model.cc.data
    Sw = model.Sw.data

    rho_c, rho_q, rho_w, rho_h = 2.55, 2.65, 1, 0.1
    rho_m = cc * (rho_c - rho_q) + rho_q
    rho_f = Sw * (rho_w - rho_h) + rho_h
    rho = Phi * (rho_f - rho_m) + rho_m

    K_c, K_q, K_w, K_h = 21, 37, 2.25, 0.04
    mu_c, mu_q = 10, 44

    K_m = ((cc * (K_c - K_q) + K_q) + (1 / (cc / K_c + (1 - cc) / K_q))) / 2
    mu_m = ((cc * (mu_c - mu_q) + mu_q) + (1 / (cc / mu_c + (1 - cc) / mu_q))) / 2
    K_f = Sw * (K_w - K_h) + K_h

    K_sat = (4 * K_m * mu_m + 3 * K_m * K_f + 4 * mu_m * K_f * Phi - 4 * K_m * mu_m * Phi) / (4 * mu_m + 3 * K_f - 3 * K_f * Phi + 3 * K_m * Phi)
    mu_sat = mu_m * (9 * K_m + 8 * mu_m) * (1 - Phi) / (9 * K_m + 8 * mu_m + 6 * (K_m + 2 * mu_m) * Phi)

    vp = ((K_sat + 4 / 3 * mu_sat) / rho)**(1/2)
    vs = (mu_sat / rho)**(1/2)

    return [vp, vs, rho]

In [ ]:
params = ['vp', 'vs', 'rho']
model_names = ['Elástico', 'Han', 'VRH', 'KT']

plot_matrix = [
    [getattr(model_elastic, param).data for param in params],
    get_Han(model_pcs),
    get_VRH(model_pcs),
    get_KT(model_pcs)
]

rows = len(plot_matrix)
cols = len(plot_matrix[0])
fig, axes = plt.subplots(rows, cols, figsize=(7*cols,2.5*rows), sharex=True, sharey=True)

for row in range(rows):
    for col in range(cols):
        img = axes[row, col].imshow(plot_matrix[row][col].T, cmap='nipy_spectral', aspect='auto')
        fig.colorbar(img)

[axes[0,i].set_title(param) for i, param in enumerate(params)]
[axes[i,0].set_ylabel(label) for i, label in enumerate(model_names)]

fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(20,6))

x = 0
[[axes[j].plot(plot_matrix[i][j][x], np.arange(len(plot_matrix[i][0][x])), label=model_names[i]) for i in range(0,4)] for j in range(len(params))]
[axes[j].invert_yaxis() for j in range(len(params))]
[axes[j].set_title(param) for j, param in enumerate(params)]
[axes[j].legend() for j in range(len(params))]

x = 500
[[axes[j+3].plot(plot_matrix[i][j][x], np.arange(len(plot_matrix[i][0][x])), label=model_names[i]) for i in range(0,4)] for j in range(len(params))]
[axes[j+3].invert_yaxis() for j in range(len(params))]
[axes[j+3].set_title(param) for j, param in enumerate(params)]
[axes[j+3].legend() for j in range(len(params))]

x = 900
[[axes[j+6].plot(plot_matrix[i][j][x], np.arange(len(plot_matrix[i][0][x])), label=model_names[i]) for i in range(0,4)] for j in range(len(params))]
[axes[j+6].invert_yaxis() for j in range(len(params))]
[axes[j+6].set_title(param) for j, param in enumerate(params)]
[axes[j+6].legend() for j in range(len(params))]

fig.tight_layout()

In [ ]:
f0 = 0.010
tn = 2000.
dt = 1

src, rec = load_setup(model=model_pcs, tn=tn, dt=dt, f0=f0)

plot_setup(model_pcs, src, rec[0], rec_step=16, param='Phi')

In [ ]:
# Forward modeling
save = True
params = ['vp-vs-rho', 'PCS_Han', 'PCS_VRH', 'PCS_KT']
V, tau, recs = {}, {}, {}

for i, param in enumerate(params):
    if i == 0:
        display(C_Matrix(model_elastic, param))
        V_, tau_, rec_ = elastic_forward(model_elastic, src, rec, param=param, save=save)
        V[param], tau[param], recs[param] = V_, tau_, rec_
    
    else:
        display(C_Matrix(model_pcs, param))
        src, rec = load_setup(model=model_pcs, tn=tn, dt=dt, f0=f0)
        V_, tau_, rec_ = elastic_forward(model_pcs, src, rec, param=param, save=save)
        V[param], tau[param], recs[param] = V_, tau_, rec_

In [ ]:
plot_matrix = np.asarray([[rec_ for rec_ in rec] for rec in recs.values()])

aspect_ratio = src.time_range.num / model_pcs.shape[0]

plt_options_model = {'cmap': 'Greys', 'extent': [0, model_pcs.domain_size[0], src.time_values[-1], 0], 'aspect':'auto'}

rows = plot_matrix.shape[0]
cols = plot_matrix.shape[1]
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(5*cols, 5*rows), sharex=True, sharey=True)

for row in range(rows):
    for col in range(cols):
        amax = max(abs(plot_matrix[row, col].data.min()), plot_matrix[row, col].data.max()) / 100

        axes[row,col].imshow(plot_matrix[row,col].data,
                             **plt_options_model,
                             vmin=-amax,
                             vmax=amax
        )
        
axes[0, 0].set_title('$V_{x}$')
axes[0, 1].set_title('$V_{z}$')
axes[0, 2].set_title('$\sigma_{xx} + \sigma_{zz}$')

axes[0, 0].set_ylabel('$V_p, V_s, \\rho$ \n\n twt$(ms)$')
[axes[i, 0].set_ylabel(f'$\\Phi$, C, $S_w$ ({params[i]}) \n\n twt$(ms)$') for i in range(1,len(params))]

[axes[3, i].set_xlabel('$x(m)$') for i in range(3)]

fig.tight_layout()
plt.show()

# fig.savefig('seismograms_marmousi.png', dpi=300)

In [ ]:
bx = model_elastic.grid.extent[0] - model_elastic.domain_size[0]
bz = model_elastic.grid.extent[1] - model_elastic.domain_size[1]
extent = [-bx/2, model_elastic.domain_size[0] + bx/2, model_elastic.domain_size[1] + bz/2, -bz/2]

snaps = np.linspace(400, src.nt-1, 4).astype('int32')
rows = len(snaps)
cols = len(tau)

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 2*rows), sharex=True, sharey=True)

for row in range(rows):
    for col in range(cols):
        snap = tau[params[col]][0].data[snaps[row]] + tau[params[col]][1].data[snaps[row]]
        axes[row, col].imshow(snap.T, cmap='gray', aspect='auto', extent=extent)

titles = ['Elastic', 'PCS (Han)', 'PCS (VRH)', 'PCS (KT)']
[axes[0,col].set_title(title) for col, title in enumerate(titles)]
[axes[row,0].set_ylabel('Depth (m)') for row in range(rows)]
[axes[-1,col].set_xlabel('Distance (m)') for col in range(cols)]

fig.tight_layout()
plt.show()

In [ ]:
fig.savefig('snaps_marmousi.png', dpi=300)